In [1]:
import pandas as pd
import numpy as np
import joblib
import shap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_percentage_error
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/Ralshatiri/Car-Price-Prediction/refs/heads/main/Dataset/Final_processed_dataset.csv')
print(df.head())
print(f"\nShape: {df.shape}")

           Make      Type      Price  Region  Gear_Type Origin    Options  \
0         Škoda    Superb   9.615805  Riyadh          0  Other  Semi Full   
1         Škoda    Superb   9.615805  Riyadh          1  Other  Semi Full   
2  Aston Martin       DB9  12.100712  Dammam          0  Saudi       Full   
3  Aston Martin       DB9  12.100712  Dammam          1  Saudi       Full   
4  Aston Martin  Vanquish  12.899220  Dammam          0  Saudi       Full   

   Engine_Size  Mileage  Negotiable  CarAge  
0          1.8   200000           0      18  
1          1.8   200000           0      18  
2          6.0    71000           0      16  
3          6.0    71000           0      16  
4          6.0    32000           0      13  

Shape: (12923, 11)


In [3]:
X = df.drop('Price', axis=1)
y = df['Price']

numerical_cols   = ["Engine_Size", "Mileage", "CarAge"]
categorical_cols = ["Make", "Type", "Region", "Origin", "Options", "Gear_Type"]
feature_names    = numerical_cols + categorical_cols + ["Negotiable"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numerical_cols),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_cols),
    ("passthrough", "passthrough", ["Negotiable"])
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f"Features: {X_train_processed.shape[1]}")

Features: 10


In [6]:
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train_processed, y_train,
    eval_set=[(X_test_processed, y_test)],
    verbose=50
)

[0]	validation_0-rmse:0.67856
[50]	validation_0-rmse:0.29884
[100]	validation_0-rmse:0.24273
[150]	validation_0-rmse:0.22220
[200]	validation_0-rmse:0.20916
[250]	validation_0-rmse:0.19904
[299]	validation_0-rmse:0.19182


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=-1, num_parallel_tree=None, ...)

In [7]:
y_pred_test  = np.exp(model.predict(X_test_processed))
y_pred_train = np.exp(model.predict(X_train_processed))
y_test_real  = np.exp(y_test)
y_train_real = np.exp(y_train)

In [8]:
def evaluate(y_true, y_pred, label=""):
    rmse = root_mean_squared_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    print(f"\n--- {label} ---")
    print(f"RMSE : {rmse:,.2f} SAR")
    print(f"R²   : {r2:.4f}")
    print(f"MAPE : {mape:.2%}")
    return rmse, r2, mape

rmse_test,  r2_test,  mape_test  = evaluate(y_test_real,  y_pred_test,  "Test set")
rmse_train, r2_train, mape_train = evaluate(y_train_real, y_pred_train, "Train set")

print(f"\nOverfitting ratio: {rmse_test / rmse_train:.3f}")


--- Test set ---
RMSE : 22,038.54 SAR
R²   : 0.8962
MAPE : 13.27%

--- Train set ---
RMSE : 17,549.76 SAR
R²   : 0.9341
MAPE : 10.72%

Overfitting ratio: 1.256


In [9]:
# create explainer
explainer   = shap.Explainer(model, X_train_processed)
shap_values = explainer(X_test_processed)

# predicted price for first test car
y_pred_price = np.exp(model.predict(X_test_processed))
print("Predicted Price:", round(y_pred_price[0], 2), "SAR")

# reasoning function
def generate_reasoning(shap_values, feature_names, index=0):
    values = shap_values.values[index]
    reasoning = []
    for feature, value in zip(feature_names, values):
        if value > 0:
            reasoning.append(f"{feature} increased the predicted price")
        elif value < 0:
            reasoning.append(f"{feature} decreased the predicted price")
    return reasoning

reasoning = generate_reasoning(shap_values, feature_names, index=0)
print("\nReasoning for the prediction:")
for r in reasoning[:5]:
    print("-", r)

 99%|===================| 2571/2585 [00:29<00:00]       

Predicted Price: 47261.82 SAR

Reasoning for the prediction:
- Engine_Size decreased the predicted price
- Mileage decreased the predicted price
- CarAge increased the predicted price
- Make increased the predicted price
- Type decreased the predicted price


In [10]:
joblib.dump(model,                    "xgboost_model.pkl")
joblib.dump(preprocessor,             "xgboost_preprocessor.pkl")
joblib.dump(feature_names,            "xgboost_feature_names.pkl")
joblib.dump(X_train_processed[:200],  "xgboost_background_data.pkl")



['xgboost_background_data.pkl']